In [2]:
import sys
import os
import xarray as xr
import numpy as np
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pyproj import Transformer
from scipy import interpolate
import matplotlib.pyplot as plt
import earthaccess

In [11]:
# Use these lines if your util folder is in the src directory
project_src_path = "/home/jovyan/Desktop/swot-surf/src"
if project_src_path not in sys.path:
    sys.path.append(project_src_path)
import util.plotting_helpers as plothelp

auth = earthaccess.login(strategy="netrc")

class SWOTAnalyzer:
    """
    A class to perform analysis and visualization of SWOT L2 Water Mask data.
    """
    def __init__(self, bounding_box, short_name="SWOT_L2_HR_Raster_D"):
        """
        Initializes the analyzer with a bounding box and searches for all
        available granules in that area.
        """
        self.short_name = short_name
        self.bounding_box = bounding_box
        self.all_granules = []
        self.granule_info = {} # Stores {granule_id: {'date': 'YYYY-MM-DD', 'granule_obj': DataGranule}}

        print(f"\n--- Initializing for bounding box: {self.bounding_box} ---")
        self._get_available_granules()
        if not self.granule_info:
            print("No valid granules found for the specified bounding box. Please check coordinates.")

    def _get_available_granules(self):
        print("Searching for all SWOT granules in the specified area (full mission duration)...")
        try:
            self.all_granules = earthaccess.search_data(
                short_name=self.short_name,
                bounding_box=self.bounding_box,
                temporal=('2023-01-01', '2025-12-31')
            )
            print(f"Found {len(self.all_granules)} granules in total.")

            if not self.all_granules:
                return

            # --- UPDATED DIAGNOSTIC PRINT ---
            print("\n--- Inspecting the first granule's RAW metadata ---")
            first_granule = self.all_granules[0]
            
            # This line will print the entire metadata dictionary
            # It will be a lot of text, so you might want to copy-paste it
            print(first_granule.raw) 
            
            print("---------------------------------------------------\n")

            temporal_attribute = 'data_start'
            for granule in self.all_granules:
                granule_id = granule.get('granule_id')
                granule_date = granule.get(temporal_attribute)
                
                if granule_id and granule_date:
                    self.granule_info[granule_id] = {
                        'date': granule_date[:10],
                        'granule_obj': granule
                    }
        except Exception as e:
            print(f"Error during Earth Data search: {e}")
            self.all_granules = []
            self.granule_info = {}

    def _prompt_and_select_granule_pair(self):
        """
        Prompts the user to select 'before' and 'after' granules from the available list.
        Returns the opened xarray Datasets for the selected pair.
        """
        if not self.granule_info:
            print("No granules available for selection. Exiting.")
        return None, None

        print("\n--- Available SWOT Granules (ID and Date) ---")
        # Sort granules by date for easier user selection
        sorted_granule_ids = sorted(self.granule_info.keys(), key=lambda k: self.granule_info[k]['date'])
        
        for i, granule_id in enumerate(sorted_granule_ids):
            info = self.granule_info[granule_id]
            print(f"  {i+1}: ID: {granule_id}, Date: {info['date']}")

        # Get user input for BEFORE granule
        while True:
            before_input = input("\nEnter the ID of the 'BEFORE' granule you want to analyze: ").strip()
            if before_input in self.granule_info:
                before_granule_obj = self.granule_info[before_input]['granule_obj']
                break
            else:
                print("Invalid ID. Please enter an ID from the list above.")
        
        # Get user input for AFTER granule
        while True:
            after_input = input("Enter the ID of the 'AFTER' granule you want to analyze: ").strip()
            if after_input in self.granule_info:
                after_granule_obj = self.granule_info[after_input]['granule_obj']
                break
            else:
                print("Invalid ID. Please enter an ID from the list above.")

        print(f"\nDownloading and opening 'BEFORE' granule: {before_input}...")
        try:
            ds_before = xr.open_dataset(earthaccess.open([before_granule_obj])[0], engine="h5netcdf")
        except Exception as e:
            print(f"Error opening 'BEFORE' granule {before_input}: {e}")
            ds_before = None

        print(f"Downloading and opening 'AFTER' granule: {after_input}...")
        try:
            ds_after = xr.open_dataset(earthaccess.open([after_granule_obj])[0], engine="h5netcdf")
        except Exception as e:
            print(f"Error opening 'AFTER' granule {after_input}: {e}")
            ds_after = None
        
        return ds_before, ds_after

    def _plot_swot_data(self, ax, ds, title, vmin, vmax):
        """
        Plots SWOT Water Surface Elevation data on a given axes.
        Returns the pcolormesh object and the WSE data array.
        """
        if ds is None:
            ax.set_title(f"{title}\n(No data available)", fontsize=14)
            ax.set_visible(False)
            return None, None

        wse = ds["wse"] + ds["height_cor_xover"]
        
        utm_zone = ds.utm_zone_num
        utm_crs = ccrs.UTM(zone=utm_zone, southern_hemisphere=False)
        transformer = Transformer.from_crs(utm_crs, ccrs.PlateCarree(), always_xy=True)
        x_utm, y_utm = wse["x"].values, wse["y"].values
        X_utm, Y_utm = np.meshgrid(x_utm, y_utm)
        X_lon, Y_lat = transformer.transform(X_utm, Y_utm)
        
        mesh = ax.pcolormesh(
            X_lon,
            Y_lat,
            wse.values,
            transform=ccrs.PlateCarree(),
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
        )
        
        ax.gridlines(draw_labels=True)
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
        ax.coastlines(resolution='10m', linewidth=1)
        ax.set_title(title, fontsize=14)
        
        return mesh, wse

    def _plot_difference(self, ax, ds_before, ds_after, title, vmin, vmax):
        """
        Calculates and plots the difference between two SWOT WSE datasets.
        Returns the pcolormesh object.
        """
        if ds_before is None or ds_after is None:
            ax.set_title(f"{title}\n(Cannot compute difference due to missing data)", fontsize=14)
            ax.set_visible(False)
            return None

        wse_before = ds_before["wse"] + ds_before["height_cor_xover"]
        wse_after = ds_after["wse"] + ds_after["height_cor_xover"]
        
        if ds_before.utm_zone_num != ds_after.utm_zone_num:
            print(f"Warning: UTM zones differ. Reprojecting 'before' to 'after's CRS for difference calculation.")
            transformer_before_to_latlon = Transformer.from_crs(
                ccrs.UTM(zone=ds_before.utm_zone_num, southern_hemisphere=False), ccrs.PlateCarree(), always_xy=True)
            transformer_after_to_latlon = Transformer.from_crs(
                ccrs.UTM(zone=ds_after.utm_zone_num, southern_hemisphere=False), ccrs.PlateCarree(), always_xy=True)
            x_before_utm, y_before_utm = wse_before["x"].values, wse_before["y"].values
            X_before_utm, Y_before_utm = np.meshgrid(x_before_utm, y_before_utm)
            lon_before, lat_before = transformer_before_to_latlon.transform(X_before_utm, Y_before_utm)
            x_after_utm, y_after_utm = wse_after["x"].values, wse_after["y"].values
            X_after_utm, Y_after_utm = np.meshgrid(x_after_utm, y_after_utm)
            lon_after, lat_after = transformer_after_to_latlon.transform(X_after_utm, Y_after_utm)
            wse_before_ll = xr.DataArray(wse_before.values, coords={'lat': lat_before[:,0], 'lon': lon_before[0,:]}, dims=['y', 'x']).rename({'y': 'lat', 'x': 'lon'})
            wse_after_ll = xr.DataArray(wse_after.values, coords={'lat': lat_after[:,0], 'lon': lon_after[0,:]}, dims=['y', 'x']).rename({'y': 'lat', 'x': 'lon'})

            try:
                wse_before_interpolated = wse_before_ll.interp(lat=wse_after_ll.lat, lon=wse_after_ll.lon, method="linear")
                wse_diff = wse_after_ll - wse_before_interpolated
            except Exception as e:
                print(f"Error during interpolation for difference: {e}.")
                ax.set_title(f"{title}\n(Error in re-gridding)", fontsize=14)
                ax.set_visible(False)
                return None
            X_lon, Y_lat = lon_after, lat_after
        else:
            try:
                wse_before_aligned = wse_before.interp(x=wse_after.x, y=wse_after.y, method="linear")
                wse_diff = wse_after - wse_before_aligned
            except Exception as e:
                print(f"Error interpolating WSE 'before' onto 'after' grid: {e}. Assuming perfect alignment.")
                if wse_before.shape == wse_after.shape and np.allclose(wse_before['x'], wse_after['x']) and np.allclose(wse_before['y'], wse_after['y']):
                    wse_diff = wse_after - wse_before
                else:
                    print("Direct difference not possible due to incompatible grids. Skipping difference plot.")
                    ax.set_title(f"{title}\n(Incompatible data grids)", fontsize=14)
                    ax.set_visible(False)
                    return None
            utm_crs_plot = ccrs.UTM(zone=ds_before.utm_zone_num, southern_hemisphere=False)
            transformer = Transformer.from_crs(utm_crs_plot, ccrs.PlateCarree(), always_xy=True)
            x_utm, y_utm = wse_after["x"].values, wse_after["y"].values
            X_utm, Y_utm = np.meshgrid(x_utm, y_utm)
            X_lon, Y_lat = transformer.transform(X_utm, Y_utm)

        mesh = ax.pcolormesh(
            X_lon,
            Y_lat,
            wse_diff.values,
            transform=ccrs.PlateCarree(),
            cmap="RdBu",
            vmin=vmin,
            vmax=vmax,
        )
        ax.gridlines(draw_labels=True)
        ax.add_feature(cfeature.LAND, facecolor='lightgray')
        ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
        ax.coastlines(resolution='10m', linewidth=1)
        ax.set_title(title, fontsize=14)
        return mesh

    def run_analysis(self):
        """Main method to run the analysis, prompting the user for input."""
        # Get the number of pairs from the user
        num_pairs = int(input("How many SWOT granule pairs (before and after) do you want to analyze? "))

        # Loop to collect data for each pair
        for i in range(num_pairs):
            print(f"\n--- Selecting granules for Pair {i+1} ---")
            ds_before, ds_after = self._prompt_and_select_granule_pair()
            
            if ds_before is not None and ds_after is not None:
                self.swot_datasets.append((ds_before, ds_after))
            else:
                print(f"Skipping Pair {i+1} due to unselected or unopenable granules.")
        
        # Proceed to plotting if any valid pairs were collected
        if not self.swot_datasets:
            print("No valid granule pairs were found or entered. Exiting.")
            return

        num_rows = len(self.swot_datasets)
        num_cols = 3
        fig = plt.figure(figsize=(24, 8 * num_rows))
        gs = fig.add_gridspec(num_rows, num_cols + 2, width_ratios=[1, 1, 1, 0.05, 0.05], wspace=0.3)
        main_location_title = input("Enter a main location title for all plots: ")

        for i, (ds_before, ds_after) in enumerate(self.swot_datasets):
            print(f"\n--- Plotting for Pair {i+1} ---")
            # Using granule metadata for dynamic titles
            before_date = ds_before.data_start_date_time[:10] if hasattr(ds_before, 'data_start_date_time') else "N/A"
            after_date = ds_after.data_start_date_time[:10] if hasattr(ds_after, 'data_start_date_time') else "N/A"

            title_before = f"WSE Before ({before_date})"
            title_after = f"WSE After ({after_date})"
            title_diff = f"WSE Difference ({before_date} vs {after_date})"

            ax1 = fig.add_subplot(gs[i, 0], projection=ccrs.PlateCarree())
            mesh1, wse1 = self._plot_swot_data(ax1, ds_before, title_before, vmin=0, vmax=10)
            ax2 = fig.add_subplot(gs[i, 1], projection=ccrs.PlateCarree())
            mesh2, wse2 = self._plot_swot_data(ax2, ds_after, title_after, vmin=0, vmax=10)
            ax3 = fig.add_subplot(gs[i, 2], projection=ccrs.PlateCarree())
            mesh3 = self._plot_difference(ax3, ds_before, ds_after, title_diff, vmin=-5, vmax=5)

            if mesh1:
                cax_wse = fig.add_subplot(gs[i, num_cols])
                cbar_wse = fig.colorbar(mesh1, cax=cax_wse, orientation='vertical')
                cbar_wse.set_label("Water Surface Elevation [m]", fontsize=12, rotation=90)
            if mesh3:
                cax_diff = fig.add_subplot(gs[i, num_cols + 1])
                cbar_diff = fig.colorbar(mesh3, cax=cax_diff, orientation='vertical')
                cbar_diff.set_label("WSE Difference [m]", fontsize=12, rotation=90)

        fig.text(x=0.5, y=0.98, s=main_location_title, fontsize=20, color='black', va='top', ha='center')
        plt.show()

# --- MAIN EXECUTION BLOCK ---
if __name__ == "__main__":
    # Get bounding box coordinates from the user
    print("--- Enter the bounding box coordinates for your area of interest ---")
    min_lon = float(input("Minimum longitude (e.g., -92.5): "))
    min_lat = float(input("Minimum latitude (e.g., 30.0): "))
    max_lon = float(input("Maximum longitude (e.g., -90.0): "))
    max_lat = float(input("Maximum latitude (e.g., 32.0): "))
    
    user_bounding_box = (min_lon, min_lat, max_lon, max_lat)

    analyzer = SWOTAnalyzer(user_bounding_box)
    analyzer.run_analysis()

--- Enter the bounding box coordinates for your area of interest ---


Minimum longitude (e.g., -92.5):  10.81748
Minimum latitude (e.g., 30.0):  14.39479
Maximum longitude (e.g., -90.0):  72.96352
Maximum latitude (e.g., 32.0):  76.68233



--- Initializing for bounding box: (10.81748, 14.39479, 72.96352, 76.68233) ---
Searching for all SWOT granules in the specified area (full mission duration)...
Found 1995 granules in total.

--- Inspecting the first granule's RAW metadata ---
Error during Earth Data search: 'DataGranule' object has no attribute 'raw'
No valid granules found for the specified bounding box. Please check coordinates.


How many SWOT granule pairs (before and after) do you want to analyze?  1



--- Selecting granules for Pair 1 ---
No granules available for selection. Exiting.
Skipping Pair 1 due to unselected or unopenable granules.


AttributeError: 'SWOTAnalyzer' object has no attribute 'swot_datasets'